In [1]:
# VFL SHAP - Prediction Notebook (Simplified & Refactored)
# ============================================================
# Clean, modular implementation for VFL model prediction with SHAP explanations

import os
import json
import joblib
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import shap
import torch
from sklearn.metrics import accuracy_score

from vfl_utils import simplify_label
from model import VFLModel, PartyMetaModel

# Configuration
MODEL_DIR = Path("model")
INPUT_DIR = Path("inputs")
OUTPUT_DIR = Path("outputs/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# 1. Helper Functions for Model Loading
# =====================================

def load_model_metadata(model_dir: Path) -> Dict:
    """Load model metadata and configuration."""
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory '{model_dir}' not found!")
    
    metadata_path = model_dir / "model_metadata.json"
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    # Convert label mapping keys to int
    metadata['label_mapping'] = {int(k): v for k, v in metadata['label_mapping'].items()}
    return metadata

def load_scalers(model_dir: Path) -> Tuple:
    """Load all party scalers."""
    scalers = []
    for i in range(1, 4):
        scaler_path = model_dir / f"scaler{i}.pkl"
        scalers.append(joblib.load(scaler_path))
    return tuple(scalers)

def load_vfl_model(model_dir: Path) -> Tuple[VFLModel, Dict]:
    """Load VFL model and return model with config."""
    model_path = model_dir / "vfl_model_best.pth"
    checkpoint = torch.load(model_path, map_location='cpu')
    config = checkpoint['model_config']
    
    model = VFLModel(
        input_dims=config['input_dims'],
        embed_dim=config['embed_dim'],
        num_classes=config['num_classes'],
        hidden_dim=config['hidden_dim']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

def load_meta_model(model_dir: Path) -> Tuple[PartyMetaModel, Dict]:
    """Load meta-model and return model with config."""
    model_path = model_dir / "meta_model_best.pth"
    checkpoint = torch.load(model_path, map_location='cpu')
    config = checkpoint['model_config']
    
    model = PartyMetaModel(
        in_dim=config['in_dim'],
        num_classes=config['num_classes'],
        hidden_dim=config['hidden_dim']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, config

def create_shap_explainer(meta_model: PartyMetaModel, background_path: Path) -> shap.KernelExplainer:
    """Create SHAP explainer for meta-model."""
    def meta_predict(x_np):
        x_t = torch.tensor(x_np, dtype=torch.float32)
        with torch.no_grad():
            logits = meta_model(x_t)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        return probs
    
    background = np.load(background_path)
    return shap.KernelExplainer(meta_predict, background)

print("✓ Helper functions defined")

✓ Helper functions defined


In [3]:
# 2. Load All Models and Configuration
# ====================================

print("="*80)
print("LOADING MODELS AND CONFIGURATION")
print("="*80)

# Load metadata
metadata = load_model_metadata(MODEL_DIR)
print(f"✓ Metadata loaded")

# Extract configuration
party_features = [
    metadata['party1_features'],
    metadata['party2_features'],
    metadata['party3_features']
]
label_mapping = metadata['label_mapping']
num_classes = metadata['num_classes']
embed_dim = metadata['embed_dim']
party_names = metadata['party_names']

print(f"✓ Configuration: {num_classes} classes, {len(party_features)} parties")

# Load scalers
scaler1, scaler2, scaler3 = load_scalers(MODEL_DIR)
scalers = [scaler1, scaler2, scaler3]
print(f"✓ Scalers loaded")

# Load models
vfl_model, vfl_checkpoint = load_vfl_model(MODEL_DIR)
print(f"✓ VFL model loaded (epoch {vfl_checkpoint['best_epoch']}, F1: {vfl_checkpoint['best_val_f1']:.4f})")

meta_model, meta_config = load_meta_model(MODEL_DIR)
print(f"✓ Meta-model loaded")

# Create SHAP explainer
shap_explainer = create_shap_explainer(meta_model, MODEL_DIR / "shap_background.npy")
print(f"✓ SHAP explainer initialized")

print("\n" + "="*80)
print("MODEL LOADING COMPLETE")
print("="*80)

LOADING MODELS AND CONFIGURATION
✓ Metadata loaded
✓ Configuration: 9 classes, 3 parties
✓ Scalers loaded
✓ VFL model loaded (epoch 19, F1: 0.6921)
✓ Meta-model loaded
✓ SHAP explainer initialized

MODEL LOADING COMPLETE


In [4]:
# 3. Helper Functions for Data Processing
# ========================================

def load_sample_data(input_dir: Path) -> pd.DataFrame:
    """Load sample CSV data."""
    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory '{input_dir}' not found!")
    
    csv_files = list(input_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {input_dir}")
    
    if len(csv_files) > 1:
        print(f"⚠ Multiple CSV files found. Using: {csv_files[0].name}")
    
    df = pd.read_csv(csv_files[0])
    # Drop unnecessary columns
    df = df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")
    return df

def validate_features(df: pd.DataFrame, party_features: List[List[str]]) -> None:
    """Validate that all required features exist in dataframe."""
    all_features = [f for party in party_features for f in party]
    missing = [f for f in all_features if f not in df.columns]
    
    if missing:
        raise KeyError(f"Missing {len(missing)} required features. First 10: {missing[:10]}")

def preprocess_data(df: pd.DataFrame, party_features: List[List[str]], 
                   scalers: List) -> List[torch.Tensor]:
    """Partition and preprocess data for each party."""
    party_tensors = []
    for party_feats, scaler in zip(party_features, scalers):
        X_raw = df[party_feats].values
        X_scaled = scaler.transform(X_raw)
        party_tensors.append(torch.tensor(X_scaled, dtype=torch.float32))
    return party_tensors

def load_labels(df: pd.DataFrame, label_mapping: Dict) -> Optional[torch.Tensor]:
    """Load and process labels if available."""
    if "label" not in df.columns:
        return None
    
    try:
        df['label_simplified'] = df['label'].apply(simplify_label)
        df['label_numeric'] = df['label_simplified'].map({v: k for k, v in label_mapping.items()})
        return torch.tensor(df['label_numeric'].values, dtype=torch.long)
    except Exception as e:
        print(f"⚠ Could not process labels: {e}")
        return None

print("✓ Data processing functions defined")

# Load and Preprocess Sample Data
print("\n" + "="*80)
print("LOADING SAMPLE DATA")
print("="*80)

sample_df = load_sample_data(INPUT_DIR)
print(f"✓ Loaded {len(sample_df)} rows")

validate_features(sample_df, party_features)
print(f"✓ All required features validated")

party_tensors = preprocess_data(sample_df, party_features, scalers)
print(f"✓ Preprocessed {len(party_tensors[0])} samples")

y_sample = load_labels(sample_df, label_mapping)
if y_sample is not None:
    print(f"✓ Labels found: {len(y_sample)} samples with ground truth")

print("="*80)

# 4. Predict on All Samples
# ==========================
print("\n" + "="*80)
print("PREDICTING ON ALL SAMPLES")
print("="*80)

n_samples = len(party_tensors[0])
results = []

print(f"Processing {n_samples} samples...")
vfl_model.eval()
meta_model.eval()

for idx in range(n_samples):
    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"  Processing sample {idx + 1}/{n_samples}...")
    
    # Get sample
    party_data = [t[idx:idx+1] for t in party_tensors]
    
    # Get true label if available
    if y_sample is not None:
        true_label_idx = int(y_sample[idx].item())
        true_label = label_mapping.get(true_label_idx, f"Class_{true_label_idx}")
    else:
        true_label_idx = None
        true_label = "UNKNOWN"
    
    # Predict
    with torch.no_grad():
        embeddings = vfl_model.get_party_embeddings(party_data)
        X_meta = torch.cat(embeddings, dim=1)
        logits = meta_model(X_meta)
        probs = torch.softmax(logits, dim=1)
        predicted_class_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0, predicted_class_idx].item()
    
    predicted_label = label_mapping.get(predicted_class_idx, f"Class_{predicted_class_idx}")
    
    # Get all probabilities
    all_probs = {label_mapping.get(i, f"Class_{i}"): float(probs[0, i].item()) 
                 for i in range(num_classes)}
    
    # Compute SHAP (party-level)
    X_meta_np = X_meta.detach().cpu().numpy()
    shap_vals_sample = shap_explainer.shap_values(X_meta_np, nsamples=50)
    
    if isinstance(shap_vals_sample, list):
        shap_vals = shap_vals_sample[predicted_class_idx][0]
    else:
        shap_vals = shap_vals_sample[0]
    
    # Aggregate to party level
    party_shap = []
    for i in range(len(party_names)):
        start = i * embed_dim
        end = (i + 1) * embed_dim
        party_shap.append(float(np.sum(np.abs(shap_vals[start:end]))))
    
    total_shap = sum(party_shap)
    party_shap_pct = [p / total_shap if total_shap > 0 else 0 for p in party_shap]
    dominant_party_idx = int(np.argmax(party_shap))
    dominant_party = party_names[dominant_party_idx]
    
    # Compute feature-level SHAP for each party
    feature_shap_contributions = {}
    
    # Create wrapper function for party-level feature SHAP
    def create_party_predictor(party_idx, fixed_embeddings):
        """Create a prediction function for a specific party's features"""
        def party_predict(X_party_np):
            """Predict using party features, keeping other parties fixed"""
            X_party_t = torch.tensor(X_party_np, dtype=torch.float32)
            batch_size = X_party_t.shape[0]
            
            vfl_model.eval()
            meta_model.eval()
            with torch.no_grad():
                # Get embedding for this party
                party_embedding = vfl_model.encoders[party_idx](X_party_t)
                
                # Combine with fixed embeddings from other parties
                all_embeddings = []
                for i, fixed_emb in enumerate(fixed_embeddings):
                    if i == party_idx:
                        all_embeddings.append(party_embedding)
                    else:
                        # Expand fixed embedding to match batch size
                        if fixed_emb.shape[0] == 1:
                            expanded_emb = fixed_emb.repeat(batch_size, 1)
                        else:
                            expanded_emb = fixed_emb[:batch_size] if fixed_emb.shape[0] >= batch_size else fixed_emb
                        all_embeddings.append(expanded_emb)
                
                X_meta_combined = torch.cat(all_embeddings, dim=1)
                logits = meta_model(X_meta_combined)
                probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            return probs
        return party_predict
    
    # Get fixed embeddings for other parties
    with torch.no_grad():
        all_embeddings_fixed = vfl_model.get_party_embeddings(party_data)
    
    # Compute feature-level SHAP for each party
    for party_idx in range(len(party_features)):
        party_name = party_names[party_idx]
        party_feat_list = party_features[party_idx]
        X_party_sample = party_data[party_idx]
        X_party_np = X_party_sample.detach().cpu().numpy()
        
        # Initialize feature contributions with zeros (will be updated if SHAP succeeds)
        feature_contribs = {feat_name: {
            "shap_value": 0.0,
            "abs_shap_value": 0.0,
            "pct_contribution": 0.0
        } for feat_name in party_feat_list}
        
        try:
            # Create better background for this party
            # Use mean-centered background with small variation
            bg_size_party = 20  # Increased for better stability
            current_feat = X_party_np[0]
            feat_mean = np.mean(current_feat)
            feat_std = np.std(current_feat)
            # Use mean ± small variation, ensuring non-negative for count features
            # Check if features are likely to be counts (based on feature names)
            is_count_feature = any('count' in f.lower() or 'packet' in f.lower() for f in party_feat_list)
            background_party = np.array([
                np.clip(
                    feat_mean + np.random.normal(0, max(feat_std * 0.1, 0.01), size=current_feat.shape),
                    a_min=0 if is_count_feature else None,
                    a_max=None
                )
                for _ in range(bg_size_party)
            ])
            
            # Ensure background has valid shape
            if background_party.shape[1] != len(party_feat_list):
                raise ValueError(f"Background shape mismatch: {background_party.shape[1]} != {len(party_feat_list)}")
            
            # Create predictor for this party
            party_predict_fn = create_party_predictor(party_idx, all_embeddings_fixed)
            
            # Test predictor function first
            test_output = party_predict_fn(X_party_np[:1])
            if test_output is None or test_output.shape[0] == 0:
                raise ValueError("Party predictor function returned invalid output")
            
            # Compute SHAP values for this party's features
            explainer_party = shap.KernelExplainer(party_predict_fn, background_party)
            shap_values_party = explainer_party.shap_values(X_party_np, nsamples=100, silent=False)
            
            # Handle multi-class SHAP output
            if isinstance(shap_values_party, list):
                if predicted_class_idx < len(shap_values_party):
                    shap_vals_party = shap_values_party[predicted_class_idx]
                    if len(shap_vals_party.shape) > 1:
                        shap_vals_party = shap_vals_party[0]
                else:
                    shap_vals_party = shap_values_party[0]
                    if len(shap_vals_party.shape) > 1:
                        shap_vals_party = shap_vals_party[0]
            else:
                shap_vals_party = shap_values_party[0] if len(shap_values_party.shape) > 1 else shap_values_party
            
            shap_vals_party = np.array(shap_vals_party).flatten()
            
            # Validate SHAP values shape
            if len(shap_vals_party) != len(party_feat_list):
                print(f"⚠ Warning: SHAP values length ({len(shap_vals_party)}) != features length ({len(party_feat_list)}) for {party_name}")
                # Pad or truncate to match
                if len(shap_vals_party) < len(party_feat_list):
                    shap_vals_party = np.pad(shap_vals_party, (0, len(party_feat_list) - len(shap_vals_party)), mode='constant')
                else:
                    shap_vals_party = shap_vals_party[:len(party_feat_list)]
            
            total_abs_shap_party = np.sum(np.abs(shap_vals_party))
            
            # Update feature-level contributions with actual SHAP values
            for feat_idx, feat_name in enumerate(party_feat_list):
                if feat_idx < len(shap_vals_party):
                    shap_val = float(shap_vals_party[feat_idx])
                    abs_shap_val = float(np.abs(shap_val))
                    pct_contrib = (abs_shap_val / total_abs_shap_party * 100.0) if total_abs_shap_party > 0 else 0.0
                    
                    feature_contribs[feat_name] = {
                        "shap_value": shap_val,
                        "abs_shap_value": abs_shap_val,
                        "pct_contribution": pct_contrib
                    }
            
            feature_shap_contributions[party_name] = feature_contribs
            
        except Exception as e:
            # Log the error for debugging
            print(f"⚠ Warning: Feature-level SHAP computation failed for {party_name}: {str(e)}")
            print(f"   Using zero values for all features in this party")
            # Store initialized dict with zeros (already initialized above)
            feature_shap_contributions[party_name] = feature_contribs
    
    # Store result with full structure
    results.append({
        "sample_id": idx,
        "true_label": true_label,
        "true_label_idx": int(true_label_idx) if true_label_idx is not None else None,
        "predicted_label": predicted_label,
        "predicted_label_idx": int(predicted_class_idx),
        "confidence": float(confidence),
        "all_probabilities": all_probs,
        "is_correct": (true_label == predicted_label) if true_label != "UNKNOWN" else None,
        "shap_explanation": {
            "party_contributions": {
                name: float(party_shap[i]) for i, name in enumerate(party_names)
            },
            "party_contributions_pct": {
                name: float(party_shap_pct[i]) for i, name in enumerate(party_names)
            },
            "dominant_party": dominant_party,
            "dominant_party_idx": int(dominant_party_idx),
            "dominant_contribution": float(party_shap[dominant_party_idx]),
            "dominant_contribution_pct": float(party_shap_pct[dominant_party_idx]),
            "total_contribution": float(total_shap),
            "feature_contributions": feature_shap_contributions
        },
        "timestamp": datetime.now().isoformat()
    })

print(f"✓ Completed predictions for {len(results)} samples")

# Calculate accuracy if labels available
if y_sample is not None:
    accuracy = sum(1 for r in results if r["is_correct"]) / len(results)
    print(f"✓ Accuracy: {accuracy:.2%}")

# 5. Save Results
# ================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save predictions summary CSV
summary_df = pd.DataFrame([{
    "sample_id": r["sample_id"],
    "true_label": r["true_label"],
    "predicted_label": r["predicted_label"],
    "confidence": r["confidence"],
    "is_correct": r["is_correct"],
    "dominant_party": r["shap_explanation"]["dominant_party"],
    "dominant_contribution_pct": r["shap_explanation"]["dominant_contribution_pct"]
} for r in results])

summary_file = OUTPUT_DIR / f"sample_predictions_summary_{timestamp}.csv"
summary_df.to_csv(summary_file, index=False)
print(f"✓ Summary: {summary_file.name}")

# Save detailed JSON
detailed_file = OUTPUT_DIR / f"sample_predictions_detailed_{timestamp}.json"
with open(detailed_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✓ Detailed: {detailed_file.name}")

# Save SHAP explanations CSV
shap_df = pd.DataFrame([{
    "sample_id": r["sample_id"],
    "predicted_label": r["predicted_label"],
    **{f"party_{i+1}_contrib": r["shap_explanation"]["party_contributions"][name]
       for i, name in enumerate(party_names)},
    **{f"party_{i+1}_contrib_pct": r["shap_explanation"]["party_contributions_pct"][name]
       for i, name in enumerate(party_names)},
    "dominant_party": r["shap_explanation"]["dominant_party"],
    "dominant_contribution_pct": r["shap_explanation"]["dominant_contribution_pct"]
} for r in results])

shap_file = OUTPUT_DIR / f"sample_shap_explanations_{timestamp}.csv"
shap_df.to_csv(shap_file, index=False)
print(f"✓ SHAP: {shap_file.name}")

# Save feature-level SHAP contributions CSV
feature_shap_data = []
for r in results:
    for party_name, party_feat_dict in r["shap_explanation"]["feature_contributions"].items():
        for feat_name, feat_data in party_feat_dict.items():
            feature_shap_data.append({
                "sample_id": r["sample_id"],
                "predicted_label": r["predicted_label"],
                "party": party_name,
                "feature_name": feat_name,
                "shap_value": feat_data.get("shap_value", 0.0),
                "abs_shap_value": feat_data.get("abs_shap_value", 0.0),
                "pct_contribution": feat_data.get("pct_contribution", 0.0)
            })

feature_shap_df = pd.DataFrame(feature_shap_data)
feature_shap_file = OUTPUT_DIR / f"sample_feature_shap_contributions_{timestamp}.csv"
feature_shap_df.to_csv(feature_shap_file, index=False)
print(f"✓ Feature SHAP: {feature_shap_file.name}")

# Save metadata JSON
metadata_output = {
    "total_samples": len(results),
    "sample_indices": [r["sample_id"] for r in results],
    "party_names": party_names,
    "label_mapping": {str(k): v for k, v in label_mapping.items()},
    "generation_timestamp": datetime.now().isoformat(),
    "model_info": {
        "num_classes": num_classes,
        "embed_dim": embed_dim
    },
    "feature_partitioning": {
        f"party{i+1}_features": feats for i, feats in enumerate(party_features)
    }
}
metadata_file = OUTPUT_DIR / f"sample_predictions_metadata_{timestamp}.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata_output, f, indent=2, ensure_ascii=False)
print(f"✓ Metadata: {metadata_file.name}")

print("\n" + "="*80)
print("PREDICTION COMPLETE")
print("="*80)
print(f"All results saved to: {OUTPUT_DIR}")
print("="*80)

✓ Data processing functions defined

LOADING SAMPLE DATA
✓ Loaded 102 rows
✓ All required features validated
✓ Preprocessed 102 samples
✓ Labels found: 102 samples with ground truth

PREDICTING ON ALL SAMPLES
Processing 102 samples...
  Processing sample 1/102...


  0%|          | 0/1 [00:00<?, ?it/s]

D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 4 iterations, i.e. alpha=8.658e-02, with an active set of 4 regressors, and the smallest cholesky pivot element being 8.429e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 4 iterations, i.e. alpha=8.658e-02, with an active set of 4 regressors, and the smallest cholesky pivot element being 5.960e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
D:\Projects\AgenticAI\.venv\Lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 8 iterations, i.e. alpha=6.756e-02, with an active set of 7 regressors, and the smallest cholesky piv

  0%|          | 0/1 [00:00<?, ?it/s]

⚠ Warning: SHAP values length (207) != features length (23) for Volume_Rate_Packet_Flow_Analyzer_Party1_SmallSet23


...

⚠ Warning: SHAP values length (207) != features length (23) for Volume_Rate_Packet_Flow_Analyzer_Party1_SmallSet23


  0%|          | 0/1 [00:00<?, ?it/s]

⚠ Warning: SHAP values length (243) != features length (27) for Timing_Direction_Protocol_Flags_Analyzer_Party2_MediumSet27


  0%|          | 0/1 [00:00<?, ?it/s]

⚠ Warning: SHAP values length (342) != features length (38) for Timing_Direction_Protocol_Flags_Analyzer_Party3_LargeSet38
✓ Completed predictions for 102 samples
✓ Accuracy: 100.00%

SAVING RESULTS
✓ Summary: sample_predictions_summary_20260206_223753.csv
✓ Detailed: sample_predictions_detailed_20260206_223753.json
✓ SHAP: sample_shap_explanations_20260206_223753.csv
✓ Feature SHAP: sample_feature_shap_contributions_20260206_223753.csv
✓ Metadata: sample_predictions_metadata_20260206_223753.json

PREDICTION COMPLETE
All results saved to: outputs\predictions
